In [ ]:
!mkdir -p ~/.keras/models/
!wget https://storage.googleapis.com/tensorflow/keras-applications/mobilenet_v2/mobilenet_v2_weights_tf_dim_ordering_tf_kernels_1.0_224_no_top.h5 -P ~/.keras/models/


In [ ]:
!ls -lh /kaggle/input/mobilenet-v2-keras-weights


In [ ]:
from tensorflow.keras.applications import MobileNetV2

weights_path = "/kaggle/input/mobilenet-v2-keras-weights/mobilenet_v2_weights_tf_dim_ordering_tf_kernels_1.0_224_no_top.h5"

base_model = MobileNetV2(
    weights=weights_path,
    include_top=False,
    input_shape=(224, 224, 3)
)

print("MobileNetV2 loaded successfully!")


In [ ]:
import os

SRC_DIR = "/kaggle/input/marin-dataset/data/zooplankton_0p5x"

for cls in os.listdir(SRC_DIR):
    cls_path = os.path.join(SRC_DIR, cls)
    if os.path.isdir(cls_path):
        print(f"\nClass: {cls}")
        for sub in os.listdir(cls_path):
            sub_path = os.path.join(cls_path, sub)
            if os.path.isdir(sub_path):
                num_images = len([
                    f for f in os.listdir(sub_path) 
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
                ])
                print(f" - {sub}: {num_images} images")


In [ ]:
import os, shutil, random
from glob import glob

BASE_DIR = "/kaggle/working/demo_rotifers"
SRC_DIR = "/kaggle/input/marin-dataset/data/zooplankton_0p5x"

# Define train/val split
SPLIT_RATIO = 0.8  

for split in ["train", "val"]:
    for cls in ["rotifers", "non_rotifers"]:
        os.makedirs(os.path.join(BASE_DIR, split, cls), exist_ok=True)

# Collect rotifer images
rotifer_images = glob(os.path.join(SRC_DIR, "rotifers", "training_data", "*.jpeg"))

# Collect non-rotifer images (all other classes)
non_rotifer_images = []
for cls in os.listdir(SRC_DIR):
    if cls != "rotifers" and os.path.isdir(os.path.join(SRC_DIR, cls, "training_data")):
        non_rotifer_images += glob(os.path.join(SRC_DIR, cls, "training_data", "*.jpeg"))

# Shuffle
random.shuffle(rotifer_images)
random.shuffle(non_rotifer_images)

def split_and_copy(images, cls_name):
    split_idx = int(len(images) * SPLIT_RATIO)
    train_imgs = images[:split_idx]
    val_imgs = images[split_idx:]

    for img in train_imgs:
        shutil.copy(img, os.path.join(BASE_DIR, "train", cls_name))
    for img in val_imgs:
        shutil.copy(img, os.path.join(BASE_DIR, "val", cls_name))

# Apply
split_and_copy(rotifer_images, "rotifers")
split_and_copy(non_rotifer_images, "non_rotifers")

print("Dataset prepared at:", BASE_DIR)
print("Rotifers train:", len(os.listdir(os.path.join(BASE_DIR, "train/rotifers"))))
print("Rotifers val:", len(os.listdir(os.path.join(BASE_DIR, "val/rotifers"))))
print("Non-rotifers train:", len(os.listdir(os.path.join(BASE_DIR, "train/non_rotifers"))))
print("Non-rotifers val:", len(os.listdir(os.path.join(BASE_DIR, "val/non_rotifers"))))

!ls /kaggle/working/demo_rotifers
!ls /kaggle/working/demo_rotifers/train
!ls /kaggle/working/demo_rotifers/val

In [ ]:
BASE_DIR = "/kaggle/working/demo_rotifers"

from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Augmentation for training (especially for rotifers)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Validation should not be augmented
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    directory=f"{BASE_DIR}/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"  # binary: rotifer vs non-rotifer
)

val_generator = val_datagen.flow_from_directory(
    directory=f"{BASE_DIR}/val",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np`

# 0 = non-rotifers, 1 = rotifers
train_classes = train_generator.classes

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_classes),
    y=train_classes
)

class_weights = dict(enumerate(class_weights))
print("Class weights:", class_weights)


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input
from tensorflow.keras.models import Model

IMG_SIZE = 224  # Already used in generators

# 1️⃣ Load MobileNetV2 base model
base_model = MobileNetV2(
    weights='imagenet',      # Use pretrained ImageNet weights
    include_top=False,       # Exclude default classifier
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False  # Freeze the base

# 2️⃣ Add custom classification head
inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.2)(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.3)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.2)(x)
outputs = Dense(1, activation='sigmoid')(x)  # Binary output

model = Model(inputs, outputs)

# 3️⃣ Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


In [ ]:
!pip install -q tensorflow-addons


In [ ]:
import os
import random
import shutil
from keras.models import Model
from keras.layers import Dense, GlobalAveragePooling2D, Dropout
from keras.applications import MobileNetV2
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# --- 1️⃣ Paths & Parameters ---
BASE_DIR = "/kaggle/working/demo_rotifers"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR = os.path.join(BASE_DIR, "val")
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
NUM_CLASSES = 2  # rotifers vs non-rotifers

# --- 2️⃣ Ensure folder names are consistent ---
# Should be exactly "rotifers" and "non-rotifers"
ROTIFER_DIR = os.path.join(TRAIN_DIR, "rotifers")
NON_ROTIFER_DIR = os.path.join(TRAIN_DIR, "non-rotifers")  # use hyphen here

# If folder has incorrect name, rename it
wrong_folder = os.path.join(TRAIN_DIR, "non_rotifers")
if os.path.exists(wrong_folder) and not os.path.exists(NON_ROTIFER_DIR):
    os.rename(wrong_folder, NON_ROTIFER_DIR)

# --- 3️⃣ Balance training data ---
rotifer_imgs = os.listdir(ROTIFER_DIR)
non_rotifer_imgs = os.listdir(NON_ROTIFER_DIR)

n_rotifer = len(rotifer_imgs)
non_rotifer_sampled = random.sample(non_rotifer_imgs, n_rotifer)

balanced_train_dir = os.path.join(BASE_DIR, "train_balanced")
os.makedirs(os.path.join(balanced_train_dir, "rotifers"), exist_ok=True)
os.makedirs(os.path.join(balanced_train_dir, "non-rotifers"), exist_ok=True)

# Copy rotifers
for img in rotifer_imgs:
    shutil.copy(os.path.join(ROTIFER_DIR, img), os.path.join(balanced_train_dir, "rotifers", img))
# Copy sampled non-rotifers
for img in non_rotifer_sampled:
    shutil.copy(os.path.join(NON_ROTIFER_DIR, img), os.path.join(balanced_train_dir, "non-rotifers", img))

print(f"✅ Balanced dataset: {len(rotifer_imgs)} rotifers, {len(non_rotifer_sampled)} non-rotifers")

# --- 4️⃣ Data generators (80/20 split) ---
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    height_shift_range=0.1,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    balanced_train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    balanced_train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
    subset='validation'
)

# --- 5️⃣ Model ---
base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224,224,3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.2)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.2)(x)
predictions = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer=Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])

# --- 6️⃣ Callbacks ---
checkpoint_path = "/kaggle/working/best_rotifer_model_balanced.h5"
early_stop = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)
checkpoint = ModelCheckpoint(filepath=checkpoint_path, monitor='val_accuracy', save_best_only=True, verbose=1)

# --- 7️⃣ Train ---
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

# --- 8️⃣ Optional fine-tuning ---
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(optimizer=Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

history_finetune = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

print("✅ Training complete. Best model saved at:", checkpoint_path)


In [ ]:
import os
import shutil

# --- Path to your balanced training folder ---
balanced_train_dir = "/kaggle/working/demo_rotifers/train_balanced"

# --- Keep only these two folders ---
allowed_folders = ["rotifers", "non-rotifers"]

# --- Iterate through all items in the folder ---
for item in os.listdir(balanced_train_dir):
    item_path = os.path.join(balanced_train_dir, item)
    
    if item not in allowed_folders:
        # Remove directories
        if os.path.isdir(item_path):
            shutil.rmtree(item_path)
            print(f"Removed extra folder: {item_path}")
        # Remove stray files
        else:
            os.remove(item_path)
            print(f"Removed stray file: {item_path}")

print("✅ Cleanup complete. Only 'rotifers' and 'non-rotifers' remain.")


In [ ]:
from pathlib import Path
import shutil

shutil.move("/kaggle/working/best_rotifer_model.h5", "/kaggle/input/marin-dataset/best_rotifer_model.h5")


In [ ]:
from pathlib import Path
import shutil
from tensorflow.keras.preprocessing.image import ImageDataGenerator

shutil.copy("/kaggle/input/best_rotifer_model_post/keras/default/1/best_rotifer_model_post.h5", "/kaggle/working/best_rotifer_model.h5")

BASE_DIR = "/kaggle/working/demo_rotifers"
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

# Validation generator (no augmentation, just rescale)
val_datagen = ImageDataGenerator(rescale=1./255)

val_generator = val_datagen.flow_from_directory(
    directory=f"{BASE_DIR}/val",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import os

# --- Step 1: Enable eager execution ---
tf.config.run_functions_eagerly(True)

# --- Step 2: Reload the model ---
model = load_model("/kaggle/input/best_rotifer_model_post/keras/default/1/best_rotifer_model_post.h5", compile=False)
print("✅ Loaded best rotifer model")

# --- Step 3: Re-compile with same settings ---
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# --- Step 4: Dataset paths ---
BASE_DIR = "/kaggle/working/demo_rotifers"
train_dir = os.path.join(BASE_DIR, "train")
val_dir = os.path.join(BASE_DIR, "val")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# --- Step 5: Recreate data generators ---
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

# --- Step 6: Class Weights (Manual Input) ---

# Get class labels from training generator
class_labels = list(train_generator.class_indices.keys())
print("Class Labels:", class_labels)

# Adjust values according to importance
class_weights = {0: 0.5222927953172676, 1: 11.714385474860336} 
print("✅ Using Manual Class Weights:", class_weights)


# --- Step 7: Define checkpoint ---
checkpoint = ModelCheckpoint(
    filepath="/kaggle/working/best_rotifer_model_continued.h5",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

# --- Step 8: Continue training ---
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5,  # adjust if needed
    class_weight=class_weights,
    callbacks=[checkpoint],
    verbose=1
)

print("✅ Training continued. Best updated model saved at /kaggle/working/best_rotifer_model_continued.h5")


In [ ]:
# --- Step 5: Evaluate + Visualize + Confidence-Based Predictions ---

import matplotlib.pyplot as plt
from keras.models import load_model
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# 1️⃣ Load the best saved model
checkpoint_path = "/kaggle/working/best_rotifer_model.h5"
model = load_model(checkpoint_path)
print("✅ Loaded best model:", checkpoint_path)

# 2️⃣ Evaluate on validation data
val_loss, val_acc = model.evaluate(val_generator)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")

# 3️⃣ Plot training history if available
if 'history' in globals() or 'history_finetune' in globals():
    plt.figure(figsize=(12,5))

    # Accuracy
    plt.subplot(1,2,1)
    if 'history' in globals():
        plt.plot(history.history['accuracy'], label='Train Accuracy')
        plt.plot(history.history['val_accuracy'], label='Val Accuracy')
    if 'history_finetune' in globals():
        plt.plot(history_finetune.history['accuracy'], label='FineTune Train Acc')
        plt.plot(history_finetune.history['val_accuracy'], label='FineTune Val Acc')
    plt.title("Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    # Loss
    plt.subplot(1,2,2)
    if 'history' in globals():
        plt.plot(history.history['loss'], label='Train Loss')
        plt.plot(history.history['val_loss'], label='Val Loss')
    if 'history_finetune' in globals():
        plt.plot(history_finetune.history['loss'], label='FineTune Train Loss')
        plt.plot(history_finetune.history['val_loss'], label='FineTune Val Loss')
    plt.title("Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.show()

# 4️⃣ Confidence-Based Predictions on a batch of validation images
CONF_THRESHOLD = 0.8  # adjust threshold if needed
val_generator.reset()
batch_images, batch_labels = next(val_generator)
pred_probs = model.predict(batch_images)

# Map class indices to names
class_indices = {v: k for k, v in val_generator.class_indices.items()}

# Apply confidence threshold
pred_labels = []
for probs in pred_probs:
    max_prob = np.max(probs)
    if max_prob < CONF_THRESHOLD:
        pred_labels.append("Not Sure")
    else:
        pred_labels.append(class_indices[np.argmax(probs)])

# Display first 8 predictions with confidence
plt.figure(figsize=(15,5))
for i in range(8):
    plt.subplot(2,4,i+1)
    plt.imshow(batch_images[i])
    plt.title(f"Pred: {pred_labels[i]}\nConf: {np.max(pred_probs[i]):.2f}")
    plt.axis('off')
plt.show()

# 5️⃣ Optional: Calculate batch accuracy ignoring 'Not Sure'
true_labels = [class_indices[np.argmax(lbl)] for lbl in batch_labels]
valid_preds = [(t, p) for t, p in zip(true_labels, pred_labels) if p != "Not Sure"]
if valid_preds:
    correct = sum(1 for t, p in valid_preds if t == p)
    accuracy = correct / len(valid_preds)
else:
    accuracy = 0
print(f"\n✅ Batch Accuracy (excluding 'Not Sure'): {accuracy*100:.2f}%")
print(f"Total 'Not Sure' predictions: {len(pred_labels) - len(valid_preds)}")

# 6️⃣ Metrics on entire validation set
val_generator.reset()
y_true, y_pred = [], []

for i in range(len(val_generator)):
    X_batch, y_batch = val_generator[i]
    probs = model.predict(X_batch)
    max_probs = np.max(probs, axis=1)
    preds = np.argmax(probs, axis=1)

    # Apply confidence threshold
    preds_conf = []
    for p, prob in zip(preds, max_probs):
        if prob < CONF_THRESHOLD:
            preds_conf.append(-1)  # -1 = Not Sure
        else:
            preds_conf.append(p)

    y_true.extend(np.argmax(y_batch, axis=1))
    y_pred.extend(preds_conf)

# Ignore 'Not Sure' (-1) for metrics
mask = np.array(y_pred) != -1
y_true_masked = np.array(y_true)[mask]
y_pred_masked = np.array(y_pred)[mask]

# Confusion matrix
cm = confusion_matrix(y_true_masked, y_pred_masked)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=val_generator.class_indices.keys(),
            yticklabels=val_generator.class_indices.keys())
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# Classification report
report = classification_report(y_true_masked, y_pred_masked,
                               target_names=val_generator.class_indices.keys())
print("\n📊 Classification Report:\n", report)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# --- Get true labels and predicted probabilities ---
val_generator.reset()
y_true = val_generator.classes
y_probs = model.predict(val_generator, verbose=1)
y_pred = np.argmax(y_probs, axis=1)

# --- Confusion Matrix ---
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

# --- Indices of False Positives (non_rotifers predicted as rotifers) ---
fp_indices = np.where((y_true == 0) & (y_pred == 1))[0]
print(f"Total False Positives: {len(fp_indices)}")

# --- Visualize a few False Positives ---
for i, idx in enumerate(fp_indices[:12]):  # show first 12
    img, label = val_generator[idx]
    plt.subplot(3, 4, i+1)
    plt.imshow(img[0])  # first image in batch
    plt.axis("off")
    plt.title(f"Pred: rotifer\nTrue: non_rotifer")
plt.show()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Evaluate model on test data
test_loss, test_acc = model.evaluate(test_generator)
print("Test Accuracy:", test_acc)
print("Test Loss:", test_loss)

# Predict on test set
y_pred = model.predict(test_generator)
y_pred_classes = np.argmax(y_pred, axis=1)

# True labels
y_true = test_generator.classes
class_labels = list(test_generator.class_indices.keys())

# Classification Report
print("Classification Report:\n", classification_report(y_true, y_pred_classes, target_names=class_labels))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - Test Set")
plt.show()


In [ ]:
# --- Save the trained model ---
model.save("/kaggle/working/best_rotifer_model.h5")
print("✅ Model saved successfully!")

# --- Load the model in a future session ---
from keras.models import load_model
loaded_model = load_model("/kaggle/working/best_rotifer_model.h5")
print("✅ Model loaded successfully!")


In [ ]:
import shutil

shutil.move("/kaggle/working/rotifer_model.h5", "/kaggle/input/rotifer_model.h5")


In [ ]:
import os
import tensorflow as tf

os.makedirs("/kaggle/working/final_model", exist_ok=True)
model = tf.keras.models.load_model("/kaggle/working/best_rotifer_model.h5")

# Save as .keras
save_path = "/kaggle/working/final_model/best_rotifer_model.keras"
model.save(save_path)
print(f"✅ Model saved at {save_path}")
